# DRF JWT Authentication

## What is JWT?

A JSON Web Token (JWT) is a compact, self-contained token that encodes a payload as a signed JSON object. Unlike DRF's `Token` model, a JWT does not require a database lookup — the server validates the signature and reads the claims directly from the token.

A JWT has three parts separated by dots:
```
header.payload.signature
```

- **Header**: algorithm and token type
- **Payload**: claims (user ID, expiry, roles, etc.)
- **Signature**: HMAC or RSA signature that proves the token was not tampered with


## Session Token vs JWT

| Aspect | DRF Token (DB) | JWT |
|--------|---------------|-----|
| Storage | Database row | Signed string |
| Stateless | No | Yes |
| Revocation | Delete the row | Requires blocklist or short TTL |
| Payload data | None | Any JSON claims |
| DB hit per request | Yes | No |
| Best for | Simple APIs | Scalable / microservice APIs |

JWTs are commonly used with short-lived **access tokens** (minutes) and longer-lived **refresh tokens** (days).


## Installation: djangorestframework-simplejwt

```bash
pip install djangorestframework-simplejwt
```

```python
# settings.py
INSTALLED_APPS += ['rest_framework_simplejwt']

REST_FRAMEWORK = {
    'DEFAULT_AUTHENTICATION_CLASSES': [
        'rest_framework_simplejwt.authentication.JWTAuthentication',
    ],
}
```


## Token Settings

```python
# settings.py
from datetime import timedelta

SIMPLE_JWT = {
    'ACCESS_TOKEN_LIFETIME':  timedelta(minutes=15),
    'REFRESH_TOKEN_LIFETIME': timedelta(days=7),
    'ROTATE_REFRESH_TOKENS':  True,     # issue a new refresh token on each refresh request
    'BLACKLIST_AFTER_ROTATION': True,   # invalidate the old refresh token (requires blacklist app)
    'ALGORITHM':              'HS256',
    'AUTH_HEADER_TYPES':      ('Bearer',),
    'AUTH_TOKEN_CLASSES':     ('rest_framework_simplejwt.tokens.AccessToken',),
}
```

`ROTATE_REFRESH_TOKENS` helps prevent refresh token reuse; pair it with the blacklist app.


## URL Configuration

```python
# urls.py
from rest_framework_simplejwt.views import (
    TokenObtainPairView,
    TokenRefreshView,
    TokenVerifyView,
)

urlpatterns += [
    path('api/token/',         TokenObtainPairView.as_view(),  name='token_obtain_pair'),
    path('api/token/refresh/', TokenRefreshView.as_view(),     name='token_refresh'),
    path('api/token/verify/',  TokenVerifyView.as_view(),      name='token_verify'),
]
```

| Endpoint | Method | Action |
|----------|--------|--------|
| `/api/token/` | POST | Exchange credentials for access + refresh tokens |
| `/api/token/refresh/` | POST | Exchange refresh token for a new access token |
| `/api/token/verify/` | POST | Check whether a token is still valid |


## Authentication Flow

```
Client                               Server
  │                                     │
  │  POST /api/token/                   │
  │  { "username": "alice",             │
  │    "password": "alicepw" }          │
  │ ─────────────────────────────────► │
  │                                     │  validate credentials
  │ ◄──────────────────────────────── │
  │  { "access":  "<15-min token>",     │
  │    "refresh": "<7-day token>" }     │
  │                                     │
  │  GET /api/books/                    │
  │  Authorization: Bearer <access>     │
  │ ─────────────────────────────────► │
  │                                     │  verify signature — no DB hit
  │ ◄──────────────────────────────── │
  │  200 OK  { ... }                    │
  │                                     │
  │  POST /api/token/refresh/           │
  │  { "refresh": "<7-day token>" }     │
  │ ─────────────────────────────────► │
  │ ◄──────────────────────────────── │
  │  { "access": "<new 15-min token>" }│
```


## Custom Claims

Add extra data to the access token payload:

```python
# users/serializers.py
from rest_framework_simplejwt.serializers import TokenObtainPairSerializer

class MyTokenObtainPairSerializer(TokenObtainPairSerializer):
    @classmethod
    def get_token(cls, user):
        token = super().get_token(user)
        token['username'] = user.username
        token['is_staff']  = user.is_staff
        return token
```

```python
# users/views.py
from rest_framework_simplejwt.views import TokenObtainPairView
from .serializers import MyTokenObtainPairSerializer

class MyTokenObtainPairView(TokenObtainPairView):
    serializer_class = MyTokenObtainPairSerializer
```

```python
# urls.py
path('api/token/', MyTokenObtainPairView.as_view(), name='token_obtain_pair'),
```


## Token Blacklisting

Enable the blacklist app to allow server-side revocation (e.g. logout):

```python
# settings.py
INSTALLED_APPS += ['rest_framework_simplejwt.token_blacklist']
```

```bash
python manage.py migrate
```

```python
# views.py
from rest_framework.views import APIView
from rest_framework.permissions import IsAuthenticated
from rest_framework.response import Response
from rest_framework import status
from rest_framework_simplejwt.tokens import RefreshToken

class LogoutView(APIView):
    permission_classes = [IsAuthenticated]

    def post(self, request):
        try:
            refresh_token = request.data['refresh']
            token = RefreshToken(refresh_token)
            token.blacklist()               # marks the refresh token as used
            return Response(status=status.HTTP_204_NO_CONTENT)
        except Exception:
            return Response(status=status.HTTP_400_BAD_REQUEST)
```


## Testing JWT Authentication

```python
from rest_framework.test import APITestCase
from django.contrib.auth.models import User

class JWTAuthTests(APITestCase):
    def setUp(self):
        self.user = User.objects.create_user(username='alice', password='alicepw')

    def get_tokens(self):
        resp = self.client.post('/api/token/', {'username': 'alice', 'password': 'alicepw'})
        self.assertEqual(resp.status_code, 200)
        return resp.data['access'], resp.data['refresh']

    def test_obtain_tokens(self):
        access, refresh = self.get_tokens()
        self.assertTrue(len(access) > 10)
        self.assertTrue(len(refresh) > 10)

    def test_protected_endpoint_with_token(self):
        access, _ = self.get_tokens()
        self.client.credentials(HTTP_AUTHORIZATION=f'Bearer {access}')
        resp = self.client.get('/api/books/')
        self.assertEqual(resp.status_code, 200)

    def test_refresh_token(self):
        _, refresh = self.get_tokens()
        resp = self.client.post('/api/token/refresh/', {'refresh': refresh})
        self.assertEqual(resp.status_code, 200)
        self.assertIn('access', resp.data)

    def test_invalid_token_rejected(self):
        self.client.credentials(HTTP_AUTHORIZATION='Bearer invalid.token.here')
        resp = self.client.get('/api/books/')
        self.assertEqual(resp.status_code, 401)
```


## Summary

- JWTs are self-contained signed tokens — the server validates the signature without a database lookup.
- `djangorestframework-simplejwt` provides `TokenObtainPairView`, `TokenRefreshView`, and `TokenVerifyView` out of the box.
- Short-lived access tokens (minutes) are sent with every request; long-lived refresh tokens (days) are used only to obtain new access tokens.
- `ROTATE_REFRESH_TOKENS` + the blacklist app enable secure logout and prevent refresh token reuse.
- Add custom claims (e.g. `username`, `is_staff`) by overriding `get_token()` in a custom serializer.
- Use `self.client.credentials(HTTP_AUTHORIZATION='Bearer <token>')` in `APITestCase` to authenticate test requests.
